# Baseline 5 — Basnet et al. (2022): ML vs DL
**Reference:** Basnet, Johnson, Doleck, *Educ. Inf. Technol.* 27(8):11499–11513, 2022. 
DOI: 10.1007/s10639-022-11068-7---
### What this reproduces
Direct ML-vs-DL comparison on the same protocol. ML side: LR, SVM, kNN, DT, NB.DL side: DNN, 1D-CNN, LSTM. Same scaler, same split, same metrics — so anydifference is from model family, not protocol.
### V2 setup notes (this re-run)
- Reads from **`dbo.tbl_graphgrued_weekly_active_base_balanced`** (V4 cohort + balanced label).
- Uses **TRUE weekly columns** `*_week1..*_week4` for windowed views (no linear scaling).
- Targets: `y_5class` (A=0..E=4) and `y_binary` (is_completed) — both already in V4.
- Stratified 8:1:1 split (seed=42), median-impute + standardize fit on train only.
### Model recipe- 5 ML + 3 DL models, same train/val/test, same scaler.
- Reports per-family means (the headline of Basnet et al.).

In [3]:
# ============================================================
# 0) IMPORTS & CONFIG
# ============================================================
import os, time, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing  import StandardScaler, LabelEncoder
from sklearn.impute         import SimpleImputer
from sklearn.metrics        import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)

SEED = 42
np.random.seed(SEED)

MODEL_BASE_TABLE = "dbo.tbl_graphgrued_weekly_active_base_ordered5"
WINDOWS = [1, 2, 4]
LABEL_ORDER_5 = ["A", "B", "C", "D", "E"]

print("Imports OK — reading from", MODEL_BASE_TABLE)


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 36, Finished, Available, Finished, False)

Imports OK — reading from dbo.tbl_graphgrued_weekly_active_base_ordered5


In [4]:
# ============================================================
# 1) LOAD MODEL BASE TABLE (V4 balanced)
# ============================================================
SAMPLE_FRACTION = 1.0
MAX_ROWS = 300_000

sdf = spark.table(MODEL_BASE_TABLE)
total_rows = sdf.count()
print(f"V4 cohort rows: {total_rows:,}")

if SAMPLE_FRACTION < 1.0:
    sdf = sdf.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SEED)

pdf = sdf.toPandas()
if len(pdf) > MAX_ROWS:
    g = pdf.groupby("risk_label_5_idx", group_keys=False)
    pdf = g.apply(lambda d: d.sample(
        n=max(1, int(len(d) * MAX_ROWS / len(pdf))),
        random_state=SEED)).reset_index(drop=True)

print(f"Working dataframe shape: {pdf.shape}")
print("\n5-class distribution after sampling:")
print(pdf["risk_label_5_idx"]
      .map({0:"A",1:"B",2:"C",3:"D",4:"E"})
      .value_counts(normalize=True).round(4)
      .reindex(LABEL_ORDER_5))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 37, Finished, Available, Finished, False)

V4 cohort rows: 100,409
Working dataframe shape: (100409, 71)

5-class distribution after sampling:
risk_label_5_idx
A    0.0222
B    0.0812
C    0.0769
D    0.6411
E    0.1785
Name: proportion, dtype: float64


In [5]:
# ============================================================
# 2) LABELS — already in V4; just alias
# ============================================================
pdf["y_5class"] = pdf["risk_label_5_idx"].astype(int)
pdf["y_binary"] = pdf["is_completed"].fillna(0).astype(int)

print("y_5class distribution:")
counts5 = pdf["y_5class"].value_counts().sort_index()
print(counts5)
min_share = counts5.min() / counts5.sum()
if min_share < 0.005:
    raise RuntimeError(
        f"Class collapse: smallest class is {min_share:.2%} < 0.5%. "
        f"Re-run 01_v4_relabel_balanced.ipynb with adjusted thresholds.")

print("\ny_binary distribution:")
print(pdf["y_binary"].value_counts(normalize=True).round(4))


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 39, Finished, Available, Finished, False)

y_5class distribution:
y_5class
0     2231
1     8156
2     7721
3    64375
4    17926
Name: count, dtype: int64

y_binary distribution:
y_binary
0    0.6692
1    0.3308
Name: proportion, dtype: float64


In [6]:
# ============================================================
# 3) FEATURE GROUPS — TRUE weekly columns + static
# ============================================================
def _by_suffix(suffix):
    return sorted([c for c in pdf.columns if c.endswith(suffix)])

WEEK_COLS = {
    1: _by_suffix("_week1"),
    2: _by_suffix("_week1") + _by_suffix("_week2"),
    3: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3"),
    4: _by_suffix("_week1") + _by_suffix("_week2") + _by_suffix("_week3") + _by_suffix("_week4"),
}

EXCLUDE = {
    "user_id", "course_id", "enroll_time",
    "is_completed", "completion_rate",
    "risk_label_5_name", "risk_label_5_idx",
    "label_D_flag", "label_E_flag",
    "y_binary", "y_5class",
    "total_4w_interactions", "post_w1_interactions", "perf_composite",
    "course_norm_score",
    "problems_passed", "course_raw_score_sum", "total_problems",
    "risk_label_5_surrogate",
}
ALL_WEEKLY = set().union(*WEEK_COLS.values())

STATIC_COLS = [
    c for c in pdf.columns
    if c not in EXCLUDE
    and c not in ALL_WEEKLY
    and pdf[c].dtype.kind in "biufc"
]

print(f"Static feature count: {len(STATIC_COLS)}")
print("Static cols:", STATIC_COLS[:20], "..." if len(STATIC_COLS) > 20 else "")
print()
for W, cols in WEEK_COLS.items():
    print(f"W={W}: {len(cols)} weekly columns up to and including week {W}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 40, Finished, Available, Finished, False)

Static feature count: 16
Static cols: ['year_of_birth', 'num_courses_enrolled', 'num_fields', 'num_resources', 'num_active_enrollments', 'course_total_videos', 'course_max_chapter', 'video_count_total_4w', 'video_watch_time_total_4w', 'chapter_coverage_ratio_raw', 'chapter_coverage_ratio', 'avg_chapter_progress_4w', 'chapter_progress_ratio_raw', 'chapter_progress_ratio', 'activity_signal_4w', 'is_active_4w'] 

W=1: 9 weekly columns up to and including week 1
W=2: 18 weekly columns up to and including week 2
W=3: 27 weekly columns up to and including week 3
W=4: 36 weekly columns up to and including week 4


In [7]:
# ============================================================
# 4) BUILD WINDOW VIEWS — leakage-safe (only weeks 1..W visible)
# ============================================================
def make_window_view(W):
    cols = STATIC_COLS + WEEK_COLS[W]
    cols = [c for c in cols if c in pdf.columns]
    Xdf  = pdf[cols].copy()
    for c in cols:
        Xdf[c] = pd.to_numeric(Xdf[c], errors="coerce").astype(float)
    return Xdf, cols

window_views = {}
for W in WINDOWS:
    Xdf, cols = make_window_view(W)
    window_views[W] = {"X": Xdf, "cols": cols}
    nan_pct = Xdf.isna().mean().mean()
    print(f"W={W}: shape={Xdf.shape}, mean NaN ratio={nan_pct:.3%}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 41, Finished, Available, Finished, False)

W=1: shape=(100409, 25), mean NaN ratio=27.326%
W=2: shape=(100409, 34), mean NaN ratio=43.076%
W=4: shape=(100409, 52), mean NaN ratio=59.910%


In [8]:
# ============================================================
# 5) STRATIFIED 8:1:1 SPLIT (per label)
# ============================================================
def split_idx(y, test_size=0.1, val_size=0.1, seed=SEED):
    idx = np.arange(len(y))
    idx_tv, idx_te = train_test_split(
        idx, test_size=test_size, stratify=y, random_state=seed)
    val_rel = val_size / (1.0 - test_size)
    idx_tr, idx_va = train_test_split(
        idx_tv, test_size=val_rel, stratify=y[idx_tv], random_state=seed)
    return idx_tr, idx_va, idx_te

splits = {}
for label in ["y_binary", "y_5class"]:
    y = pdf[label].values
    splits[label] = split_idx(y)
    n_tr, n_va, n_te = (len(splits[label][0]), len(splits[label][1]), len(splits[label][2]))
    print(f"{label}: train={n_tr}, val={n_va}, test={n_te}")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 65, Finished, Available, Finished, False)

y_binary: train=80327, val=10041, test=10041
y_5class: train=80327, val=10041, test=10041


In [9]:
# ============================================================
# 6) METRIC HELPERS — EarlyRisk-MOOC metric set
# ============================================================
def evaluate_classifier(y_true, y_pred, y_proba=None, classes=None, name=""):
    res = {"name": name}
    res["accuracy"]    = accuracy_score(y_true, y_pred)
    res["macro_f1"]    = f1_score(y_true, y_pred, average="macro", zero_division=0)
    res["weighted_f1"] = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    if classes is None:
        classes = sorted(list(set(y_true) | set(y_pred)))
    per_class_f1 = f1_score(y_true, y_pred, average=None,
                            labels=classes, zero_division=0)
    for cls, f in zip(classes, per_class_f1):
        res[f"f1_{cls}"] = float(f)
    if y_proba is not None and len(classes) >= 2:
        try:
            if len(classes) == 2:
                res["auc"] = roc_auc_score(y_true, y_proba[:, 1])
            else:
                res["auc_ovr"] = roc_auc_score(
                    y_true, y_proba, multi_class="ovr",
                    labels=classes, average="macro")
        except Exception as e:
            res["auc_error"] = str(e)
    return res


def flatten_result(res, name, classes_order, runtime, W, label_col):
    out = {"name": name, "label": label_col, "window": W,
           "runtime_sec": round(runtime, 2)}
    for k, v in res.items():
        if k.startswith("f1_") and k != "f1_score":
            try:
                idx = int(k.split("_")[1])
                out[f"f1_{classes_order[idx]}"] = v
            except (ValueError, IndexError):
                out[k] = v
        elif k in ("accuracy","macro_f1","weighted_f1","auc","auc_ovr"):
            out[k] = v
    return out


print("Metric helpers ready")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 66, Finished, Available, Finished, False)

Metric helpers ready


In [10]:
# ============================================================
# 7) IMPUTATION + SCALING (fit on TRAIN only)
# ============================================================
def fit_impute_scale(X_tr, X_te, X_va=None):
    imp = SimpleImputer(strategy="median").fit(X_tr)
    X_tr_i = imp.transform(X_tr)
    X_te_i = imp.transform(X_te)
    sc = StandardScaler().fit(X_tr_i)
    out = (sc.transform(X_tr_i), sc.transform(X_te_i))
    if X_va is not None:
        out = out + (sc.transform(imp.transform(X_va)),)
    return out

print("fit_impute_scale ready")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 67, Finished, Available, Finished, False)

fit_impute_scale ready


In [11]:
# ============================================================
# 8) BASNET — ML vs DL benchmark
# ============================================================
from sklearn.linear_model   import LogisticRegression
from sklearn.svm            import LinearSVC
from sklearn.neighbors      import KNeighborsClassifier
from sklearn.tree           import DecisionTreeClassifier
from sklearn.naive_bayes    import GaussianNB
from sklearn.calibration    import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier

TORCH_AVAILABLE = False
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
    TORCH_AVAILABLE = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    pass


if TORCH_AVAILABLE:
    class DNN(nn.Module):
        def __init__(self, in_dim, n_cls):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(128, 64),     nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, 32),      nn.ReLU(),
                nn.Linear(32, n_cls))
        def forward(self, x): return self.net(x)
    class CNN1D(nn.Module):
        def __init__(self, in_dim, n_cls):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv1d(1, 16, 3, padding=1), nn.ReLU(),
                nn.Conv1d(16, 32, 3, padding=1), nn.ReLU(),
                nn.AdaptiveAvgPool1d(8))
            self.head = nn.Sequential(nn.Flatten(),
                nn.Linear(32*8, 64), nn.ReLU(), nn.Linear(64, n_cls))
        def forward(self, x):
            return self.head(self.conv(x.unsqueeze(1)))
    class LSTMTab(nn.Module):
        def __init__(self, in_dim, n_cls, hidden=64):
            super().__init__()
            self.lstm = nn.LSTM(in_dim, hidden, batch_first=True)
            self.head = nn.Linear(hidden, n_cls)
        def forward(self, x):
            _, (h, _) = self.lstm(x.unsqueeze(1))
            return self.head(h.squeeze(0))

    def train_dl(model, X, y, epochs=15, batch=512, lr=1e-3):
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        crit = nn.CrossEntropyLoss()
        ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                           torch.tensor(y, dtype=torch.long))
        dl = DataLoader(ds, batch_size=batch, shuffle=True)
        model = model.to(DEVICE)
        for _ in range(epochs):
            model.train()
            for a, b in dl:
                a, b = a.to(DEVICE), b.to(DEVICE)
                opt.zero_grad(); crit(model(a), b).backward(); opt.step()
        return model

    def predict_dl(model, X):
        model.eval()
        with torch.no_grad():
            p = torch.softmax(model(torch.tensor(X, dtype=torch.float32).to(DEVICE)), dim=1).cpu().numpy()
        return p.argmax(1), p


def make_ml_models():
    return {
        "LogReg": LogisticRegression(max_iter=1000, n_jobs=-1, random_state=SEED),
        "SVM":    CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=SEED),
                                          cv=3, method="sigmoid"),
        "kNN":    KNeighborsClassifier(n_neighbors=15, n_jobs=-1),
        "DT":     DecisionTreeClassifier(max_depth=12, random_state=SEED),
        "NB":     GaussianNB(),
    }


all_results = []
for label_col, classes in [("y_binary", [0,1]), ("y_5class", LABEL_ORDER_5)]:
    idx_tr, idx_va, idx_te = splits[label_col]
    y_all = pdf[label_col].values
    n_cls = len(classes)
    for W in WINDOWS:
        X = window_views[W]["X"].values
        X_tr_s, X_te_s = fit_impute_scale(X[idx_tr], X[idx_te])

        # ML
        for name, m in make_ml_models().items():
            print(f"[{label_col}] W={W} ML/{name} ...")
            t0 = time.time(); m.fit(X_tr_s, y_all[idx_tr]); rt = time.time()-t0
            y_pred = m.predict(X_te_s)
            y_proba = m.predict_proba(X_te_s) if hasattr(m, "predict_proba") else None
            res = evaluate_classifier(y_all[idx_te], y_pred, y_proba,
                                       classes=list(range(n_cls)), name=name)
            row = flatten_result(res, name, classes, rt, W, label_col)
            row["family"] = "ML"
            all_results.append(row)

        # DL
        for name in ["DNN", "CNN1D", "LSTM"]:
            print(f"[{label_col}] W={W} DL/{name} ...")
            t0 = time.time()
            if TORCH_AVAILABLE:
                if   name == "DNN":   model = DNN(X_tr_s.shape[1], n_cls)
                elif name == "CNN1D": model = CNN1D(X_tr_s.shape[1], n_cls)
                else:                  model = LSTMTab(X_tr_s.shape[1], n_cls)
                model = train_dl(model, X_tr_s, y_all[idx_tr])
                y_pred, y_proba = predict_dl(model, X_te_s)
            else:
                clf = MLPClassifier(hidden_layer_sizes=(128,64), max_iter=60,
                                    random_state=SEED, early_stopping=True).fit(X_tr_s, y_all[idx_tr])
                y_pred, y_proba = clf.predict(X_te_s), clf.predict_proba(X_te_s)
            rt = time.time()-t0
            res = evaluate_classifier(y_all[idx_te], y_pred, y_proba,
                                       classes=list(range(n_cls)), name=name)
            row = flatten_result(res, name, classes, rt, W, label_col)
            row["family"] = "DL"
            all_results.append(row)

res_df = pd.DataFrame(all_results)
print("\n=== Basnet et al. 2022 — V4 ML vs DL ===")
print(res_df.round(4).to_string(index=False))
fam = (res_df.groupby(["label","window","family"])
              .agg(macro_f1_mean=("macro_f1","mean"),
                   accuracy_mean=("accuracy","mean"),
                   runtime_mean=("runtime_sec","mean"))
              .round(4).reset_index())
print("\n=== Family means: ML vs DL ===")
print(fam.to_string(index=False))
res_df.to_csv("/tmp/results_basnet_2022.csv", index=False)
fam.to_csv("/tmp/results_basnet_2022_familymeans.csv", index=False)
print("\nSaved -> /tmp/results_basnet_2022*.csv")


StatementMeta(, 1640044a-dd4c-45ea-af77-87f2048fade8, 68, Finished, Available, Finished, False)

[y_binary] W=1 ML/LogReg ...


[y_binary] W=1 ML/SVM ...


[y_binary] W=1 ML/kNN ...


[y_binary] W=1 ML/DT ...


...

[y_5class] W=4 ML/SVM ...


[y_5class] W=4 ML/kNN ...


[y_5class] W=4 ML/DT ...


[y_5class] W=4 ML/NB ...


[y_5class] W=4 DL/DNN ...
[y_5class] W=4 DL/CNN1D ...
[y_5class] W=4 DL/LSTM ...

=== Basnet et al. 2022 — V4 ML vs DL ===
  name    label  window  runtime_sec  accuracy  macro_f1  weighted_f1   f1_0   f1_1    auc family   f1_A   f1_B   f1_C   f1_D   f1_E  auc_ovr
LogReg y_binary       1        17.64    0.8311    0.7903       0.8216 0.8828 0.6978 0.8861     ML    NaN    NaN    NaN    NaN    NaN      NaN
   SVM y_binary       1       143.63    0.8116    0.7574       0.7962 0.8720 0.6427 0.8848     ML    NaN    NaN    NaN    NaN    NaN      NaN
   kNN y_binary       1       123.75    0.8912    0.8738       0.8897 0.9207 0.8268 0.9480     ML    NaN    NaN    NaN    NaN    NaN      NaN
    DT y_binary       1         6.82    0.9002    0.8839       0.8987 0.9274 0.8405 0.9431     ML    NaN    NaN    NaN    NaN    NaN      NaN
    NB y_binary       1         6.09    0.3909    0.3512       0.2968 0.1906 0.5117 0.7468     ML    NaN    NaN    NaN    NaN    NaN      NaN
   DNN y_binary       1  